###Data Reading JSON

In [0]:
df_json=spark.read.format("json").option("inferSchema",True).option("header",True).option("multiLine",False).load("/Volumes/workspace/databricks_pyspark_tutorial/mydatavolume/drivers.json")

In [0]:
df_json.display()

####using dbutils to see what files inside the volumes

In [0]:
dbutils.fs.ls("/Volumes/workspace/databricks_pyspark_tutorial/mydatavolume")

###Data Reading

In [0]:
df=spark.read.format("csv").option("inferSchema",True).option("header",True).load("/Volumes/workspace/databricks_pyspark_tutorial/mydatavolume/BigMart Sales.csv")

In [0]:
df.display()

In dataframe if we didn't do inferschema as true , then manually write the schema .This ca be done in 2 methods ,
DDL and StrucType()

Schema Definition

In [0]:
df.printSchema()

###DDL SCHEMA

In [0]:
my_ddl_schema='''
Item_identifier string,
Item_Weight string,
Item_Fat_Content string,
Item_Visibility double,
Item_Type string,
Item_MRP double,
Outlet_Identifier string,
Outlet_Establishment_Year integer,
Outlet_Size string,
Outlet_Location_Type string,
Outlet_Type string,
Item_Outlet_Sales double'''

In [0]:
df=spark.read.format('csv').option('header',True).schema(my_ddl_schema).load("/Volumes/workspace/databricks_pyspark_tutorial/mydatavolume/BigMart Sales.csv")


In [0]:
df.display()

In [0]:
df.printSchema()


###StructType() Schema

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
my_struct_schema=StructType([
                             StructField('Item_Identifier',StringType(),True)  ,
                             StructField('Item_Weight',StringType(),True), StructField('Item_Fat_Content',StringType(),True), StructField('Item_Visibility',StringType(),True), StructField('Item_MRP',StringType(),True), StructField('Outlet_Identifier',StringType(),True), StructField('Outlet_Establishment_Year',StringType(),True), StructField('Outlet_Size',StringType(),True), StructField('Outlet_Location_Type',StringType(),True), StructField('Outlet_Type',StringType(),True), StructField('Item_Outlet_Sales',StringType(),True)
])

In [0]:
df=spark.read.format("csv").schema(my_struct_schema).option('header',True).load("/Volumes/workspace/databricks_pyspark_tutorial/mydatavolume/BigMart Sales.csv")


In [0]:
df.printSchema()

###Running the normal schema generated through inferSchema to do further transformations\
run the cell 7 and run cell 23

In [0]:
df.display()

###Transformations

###SELECT

In [0]:
df.select('Item_Identifier','Item_Weight','Item_Fat_Content').display()

In [0]:
df.select(col('Item_Identifier'),col('Item_Weight'),col('Item_Fat_Content')).display()

###ALIAS 
cannot use Alias without an column object

In [0]:
df.select(col('Item_Identifier').alias('Item_ID')).display()

In [0]:
df.display()

###FILTER/WHERE

####Scenario 1
In Item_Fat_Content pick only Regular\
Difference bw select and filter transformations is ,Select does selection of columns from dataframe.\
Whereas , filter does the selection of rows based on the condition given.

In [0]:
df.filter(col('Item_Fat_Content')=='Regular').display()

####Scenario 2
Filtering 2 cols at a time

In [0]:
df.filter( (col('Item_Type')=='Soft Drinks')& (col('Item_Weight')<10) ).display()

####Scenario 3
fetch the data with Tier1 or Tier2 and Outlet size is Null

In [0]:
df.filter((col('Outlet_Location_Type').isin('Tier 1','Tier 2')& (col("Outlet_Size").isNull()))).display()

###withColumnRenamed

In [0]:
df.withColumnRenamed('Item_Weight','Item_wt').display()

###withColumn
creating the new column and helps modifying the column

####Scenario 1
Creating new column
if you want to make constant value in the col , you have to function called lit()
Syntax of withColumn:
withColumn('New/Existing column name')  (important)

In [0]:
df=df.withColumn('flag',lit('new'))

In [0]:
df.display()

In [0]:
df.withColumn('multiply',col('Item_Weight')*col('Item_MRP')).display()

####Scenario 2
Modifying the existing columns

In [0]:
df.withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),'Regular',"Reg"))\
    .withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),'Low Fat','LF')).display()

###Type Casting
Here in the below example we use withColumn() because we are modifying the data .

In [0]:
df=df.withColumn('Item_Weight',col('Item_Weight').cast(StringType()))

In [0]:
df.printSchema()

###Sort/orderBy

####Scene 1
Based on a particular column we sorting the dataframe in descending order or ascending order.
Here no need of using withColumn as we used in .cast() function.

In [0]:
df.sort(col('Item_Weight').desc()).display()

####Scene2


In [0]:
df.sort(col('Item_Visibility').asc()).display()

####Scene3
Perform sorting based on multiple columns
True :1 ,False:0

In [0]:
df.sort(["Item_Weight","Item_Visibility"],ascending=[False,False]).display()

####Scene 4


In [0]:
df.sort(["Item_Weight","Item_Visibility"],ascending=[False,True]).display()

###Limit

In [0]:
df.limit(10).display()

###DROP

####Scene1
Dropping single column

In [0]:
df.drop(col('Item_Visibility')).display()

####Scene2
Dropping Multiple colums

In [0]:
df.drop('Item_Visibility','Item_Type').display()

###DROP_DUPLICATES
If we have duplicate row in dataframe , we can do remove them with this.
if we dont want duplicates in a particular column , we can perform using this.

####Scene1

In [0]:
df.dropDuplicates().display()

####Scene 2
Below we will only get 16 rows.


In [0]:
df.dropDuplicates(subset=['Item_Type']).display()

In [0]:
df.distinct().display()

###UNION and UNION BYNAME

####Preparing DataFrames

In [0]:
data1=[
    ('1','kad'),
    ('2','sid')
]
schema1='id STRING, name STRING'
df1=spark.createDataFrame(data1,schema1)

data2=[
    ('3','rahul'),
    ('4','jas')
]
schema2='id STRING, name STRING'
df2=spark.createDataFrame(data2,schema2)

In [0]:
df1.display()

In [0]:
df2.display()

####UNION

In [0]:
df1.union(df2).display()

####UNIONBYNAME

#####Preparing the dataframe again changing the order.

In [0]:
data1=[
    ('kad','1'),
    ('sid','2')
]
schema1='name String, id String'
df1=spark.createDataFrame(data1,schema1)


In [0]:
df1.display()

In [0]:
df1.unionByName(df2).display()

###String Functions

####Initcap()

In [0]:
from pyspark.sql.functions import *
df.select(initcap('Item_Type')).display()

In [0]:
df.select(lower('Item_Type')).display()

In [0]:
df.select(upper('Item_Type').alias('upper_Item_Type')).display()


###DATE FUNCTIONS

####CURRENT_DATE

In [0]:
df=df.withColumn('current_date',current_date())
df.display()

####DATE_ADD

In [0]:
df=df.withColumn('week_after',date_add('current_date',7))
df.display()

####DATE_SUB

In [0]:
df=df.withColumn('week_before',date_sub('current_date',7))
df.display()

#####Date_sub with date_add

In [0]:
df=df.withColumn('week_before_date_add',date_add('current_date',-7))
                 
df.display()

###DATE_DIFF

In [0]:
df=df.withColumn('difference_date',datediff('current_date','week_after'))
df.display()

###DATE_FORMAT

In [0]:
df=df.withColumn('week_before',date_format('week_before','dd-MM-yyyy'))
df.display()

### HANDLING NULLS
dropping nulls,
filling nulls

####Scene1: Dropping Nulls
dropna means dropnotavailable
in dropna we have 2 options 
    -all , if all the record(row) is empty only that row is dropp.
    -any  ,if any of the records is empty that row is dropped.

In [0]:
df.dropna('all').display()

In [0]:
df.dropna('any').display()

In [0]:
df.dropna(subset=['Outlet_Size']).display()
#this only drops the null values which is present in the Outlet_Size


####Fill the Nulls

In [0]:
df.fillna('not available').display()


In [0]:
df.fillna('NA',subset=['Outlet_Size']).display()

###SPLIT and INDEXING

####SPLIT

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')).display()

####INDEXING

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')[1]).display()

###EXPLODE

In [0]:
df_new=df.withColumn('Outlet_Type',split('Outlet_Type',' '))

df_new.display()

In [0]:
df_new.withColumn('Outlet_Type',explode('Outlet_Type')).display()

###ARRAY_CONTAINS
returns boolean value.

In [0]:
df_new.display()

In [0]:
df_new.withColumn('Type1_flag',array_contains('Outlet_Type','Type1')).display()

###GROUP BY
Grouping Item_type column and finding the Item_MRP based on each Item_Type.

In [0]:
df.display()

In [0]:
df.groupBy('Item_Type').agg(sum("Item_MRP")).display()

In [0]:
df.groupBy('Item_Type').agg(avg('Item_MRP')).display()
#finding the avg Item_MRP based on Item_Type

####Scene
grouping 2 or more columns

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP').alias('Total_MRP')).display()

####Scene2
Applying 2 or more aggreations in group by

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP'),avg('Item_MRP')).display()

###COLLECT_LIST
groups and holds the values in a list \
group concat in mysql

In [0]:
data1=[
    ('user1','book1'),
    ('user1','book2'),
    ('user2','book2'),
    ('user2','book4'),
    ('user3','book1')
]
schema='user string, book string'
df_book=spark.createDataFrame(data1,schema)
df_book.display()


In [0]:
df_book.groupBy('user').agg(collect_list('book')).display()

###PIVOT
used in excel
uses groupby

In [0]:
df.groupBy('Item_Type').pivot('Outlet_Size').agg(avg('Item_MRP')).display()

###WHEN-OTHERWISE
similar to case-when in sql\
if we want to create a new column based on the condition use this function

####Scene1

In [0]:
df=df.withColumn('VegOrNonVeg',\
              when(col('Item_Type')=='Meat','Non-Veg')\
              .otherwise('Veg'))
              
df.display()

####Scene2
create a new column 'Veg_rate_flag'
If Item_type is veg and Item_mrp is >100 comes under veg_expensive and if veg and rate less than 100 comes under veg_inexpensive,otherwise it is non veg.

In [0]:

df.withColumn('Veg_rate_flag',\
              when( (col('VegOrNonVeg')=='Veg') & (col('Item_MRP')<100),'Veg_Inexpensive')\
                  .when((col('VegOrNonVeg')=='Veg') & (col('Item_MRP')>100),'Veg_expensive')\
                      .otherwise('Non-Veg')
                  
).display()
                  

###JOINS